<a href="https://colab.research.google.com/github/ttn64681/Bird-Audio-Classification/blob/main/ast.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ── Install required libraries ──────────────────────────────────────
!pip install -q transformers datasets evaluate torch pandas matplotlib seaborn scikit-learn kagglehub librosa fvcore

In [ ]:
# ── Imports ──────────────────────────────────────────────────────────
import torch
import collections
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os
import librosa
import time
from IPython.display import Image, display, Audio

from transformers import (
    ASTFeatureExtractor,   # was AutoFeatureExtractor
    ASTForAudioClassification,  # was AutoModelForAudioClassification
    TrainingArguments,
    Trainer,
    pipeline
)
import evaluate
from datasets import Dataset, DatasetDict, load_from_disk
from sklearn.metrics import confusion_matrix, classification_report

# Print PyTorch and device information
print(f"PyTorch version : {torch.__version__}")
print(f"CUDA available  : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU             : {torch.cuda.get_device_name(0)}")


In [ ]:
!cp -r /content/drive/MyDrive/HuBERT/BirdCLEF/ /content/BirdCLEF_local/

In [ ]:
# ── Mount Google Drive & Setup Labels ───────────────────────────────
from google.colab import drive
drive.mount('/content/drive')
# Check what is actually inside that folder
print(os.listdir('/content/drive/MyDrive/HuBERT/BirdCLEF/'))

# Rebuild the label dictionaries globally
DATA_PATH = '/content/BirdCLEF_local/'
species = sorted([d for d in os.listdir(DATA_PATH) if os.path.isdir(os.path.join(DATA_PATH, d))])
labels = species
label2id = {name: i for i, name in enumerate(species)}
id2label = {i: name for i, name in enumerate(species)}
print(f"Unique labels: {labels}")

In [ ]:
# ── Load AST model & Extractor ───────────────────────────────────────

# Use MIT's AST base model
MODEL_CHECKPOINT = "MIT/ast-finetuned-audioset-10-10-0.4593"

# Use AutoFeatureExtractor instead of AutoTokenizer (for compatibility with audio, not text)
feature_extractor = ASTFeatureExtractor.from_pretrained(MODEL_CHECKPOINT)
print(f"Feature extractor loaded: {MODEL_CHECKPOINT}")

In [ ]:
# ── AST Dataset Generator ────────────────────────────────────────────

def ast_audio_generator(data_path, limit_per_class=500):
    for bird in species:
        print(f"Extracting AST spectrograms for: {bird}")
        folder_path = os.path.join(data_path, bird)
        files = [f for f in os.listdir(folder_path) if f.lower().endswith(('.ogg', '.wav', '.mp3', '.m4a'))]

        count = 0
        for f in files:
            if count >= limit_per_class: break
            file_path = os.path.join(folder_path, f)
            try:
                # 1. Load the audio
                audio, _ = librosa.load(file_path, sr=16000)

                # 2. AST PREPROCESSING (Creates 2D Spectrogram Tensors)
                encoded = feature_extractor(
                    audio,
                    sampling_rate=16000,
                    return_tensors="np"
                )

                # 3. Yield to the dataset builder
                yield {
                    "input_values": encoded["input_values"][0],
                    "label": label2id[bird]
                }
                count += 1
            except Exception as e:
                pass # Skip corrupted files silently to keep output clean

# ── RUN THE GENERATOR & SAVE ─────────────────────────────────────────
# WARNING: If you already ran this and saved to Drive, skip building and just load below!
print("Building AST dataset file-by-file...")
ast_dataset = Dataset.from_generator(
    ast_audio_generator,
    gen_kwargs={"data_path": DATA_PATH}
)

print("Splitting into Train, Val, and Test...")
first_split = ast_dataset.train_test_split(test_size=0.1, seed=42)
second_split = first_split['train'].train_test_split(test_size=0.222, seed=42)

final_datasets = DatasetDict({
    'train': second_split['train'],
    'validation': second_split['test'],
    'test': first_split['test']
})

AST_SAVE_PATH = '/content/drive/MyDrive/HuBERT_Thai/ast_bird_datasets_official'
final_datasets.save_to_disk(AST_SAVE_PATH)
print(f"Success! AST dataset saved to: {AST_SAVE_PATH}")

In [ ]:
AST_SAVE_PATH = '/content/drive/MyDrive/HuBERT_Thai/ast_bird_datasets_official'
print(os.listdir('/content/drive/MyDrive/HuBERT_Thai/ast_bird_datasets_official/'))


In [ ]:
# ── Load and Subsample Datasets ──────────────────────────────────────

# 1. Load the Master Datasets (the full 500-per-bird set)
print(f'Loading AST Datasets from {AST_SAVE_PATH}...')
loaded_datasets = load_from_disk(AST_SAVE_PATH)

train_dataset = loaded_datasets['train']
val_dataset = loaded_datasets['validation']
test_dataset = loaded_datasets['test']

# # Scale down to 1050/300/150 for the exact same experiment as HuBERT
# train_dataset = train_dataset.shuffle(seed=42).select(range(1050))
# val_dataset   = val_dataset.shuffle(seed=42).select(range(300))
# test_dataset  = test_dataset.shuffle(seed=42).select(range(150))

print(f"--- Scaled AST Dataset Sizes ---")
print(f"Train: {len(train_dataset)}")
print(f"Val:   {len(val_dataset)}")
print(f"Test:  {len(test_dataset)}")

# ── Metric functions for measuring accuracy ──────────────────────────
precision_metric = evaluate.load("precision")
recall_metric = evaluate.load("recall")
accuracy_metric = evaluate.load("accuracy")
f1_metric = evaluate.load("f1")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=1)

    acc = accuracy_metric.compute(predictions=predictions, references=labels)
    f1 = f1_metric.compute(predictions=predictions, references=labels, average="macro")
    prec = precision_metric.compute(predictions=predictions, references=labels, average="macro")
    rec = recall_metric.compute(predictions=predictions, references=labels, average="macro")

    return {
        "accuracy": acc["accuracy"],
        "f1": f1["f1"],
        "precision": prec["precision"],
        "recall": rec["recall"]
    }

In [ ]:
# ── Baseline Evaluation (Untrained Head) ─────────────────────────────
print("\n" + "="*50)
print("Evaluating Base AST (Untrained Classification Head)")
print("="*50)

base_model = ASTForAudioClassification.from_pretrained(
    MODEL_CHECKPOINT,
    num_labels=10,
    ignore_mismatched_sizes=True
)

base_trainer = Trainer(
    model=base_model,
    args=TrainingArguments(output_dir="/tmp/base_eval", per_device_eval_batch_size=16, report_to="none"),
    eval_dataset=test_dataset,
    compute_metrics=compute_metrics,
)

base_results = base_trainer.predict(test_dataset)
print(f"  Base Accuracy   : {base_results.metrics['test_accuracy']:.4f}")
print(f"  Base F1 Score   : {base_results.metrics['test_f1']:.4f}")

In [ ]:
# Clean GPU memory (to wipe base model) before fine-tuning
del base_model
del base_trainer
import gc
gc.collect()
torch.cuda.empty_cache()
print("Cleared GPU memory for fine-tuning!")

In [ ]:
# ── Fine-Tuning AST ──────────────────────────────────────────────────
model = ASTForAudioClassification.from_pretrained(
    MODEL_CHECKPOINT,
    num_labels=10,
    ignore_mismatched_sizes=True # Required for AST
)

# Keep these identical to HuBERT for the research comparison
training_args = TrainingArguments(
    output_dir="/content/ast_training_checkpoints",
    num_train_epochs=10,
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=16,
    seed=42,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
    save_total_limit=1,
    report_to="none",
)

trainer_birdclef = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics,
)

In [ ]:
trainer_birdclef.train()

In [ ]:
# After training finishes:
print("Saving the BEST model permanently to Google Drive...")
# This only saves the final weights (~300MB), not the 10 epochs of junk
trainer_birdclef.save_model("/content/drive/MyDrive/HuBERT_Thai/ast_model_500_10_8_16")
print("Done!")

In [ ]:
# ── Research Analytics & Final Results ───────────────────────────────
num_params = sum(p.numel() for p in model.parameters())
print(f"Total Parameters: {num_params / 1e6:.2f} Million")

if len(trainer_birdclef.state.log_history) > 0:
    train_metrics = trainer_birdclef.state.log_history[-1]
    if 'train_samples_per_second' in train_metrics:
        print(f"Training throughput: {train_metrics['train_samples_per_second']:.2f} samples/sec")

start_inference = time.time()
results = trainer_birdclef.predict(test_dataset)
end_inference = time.time()

total_time = end_inference - start_inference
latency_per_sample = (total_time / len(test_dataset)) * 1000

print("\n" + "=" * 50)
print("AST Fine-Tuned — Final Test Results")
print("=" * 50)
print(f"  Accuracy   : {results.metrics['test_accuracy']:.4f}")
print(f"  F1 Score   : {results.metrics['test_f1']:.4f}")
print(f"  Precision  : {results.metrics['test_precision']:.4f}")
print(f"  Recall     : {results.metrics['test_recall']:.4f}")
print("-" * 50)
print(f"Total Test Inference Time: {total_time:.2f} seconds")
print(f"Average Latency: {latency_per_sample:.2f} ms per bird call")

In [ ]:
# ── MACs & FLOPs ───────────────────────────────

from fvcore.nn import FlopCountAnalysis
import torch

# 1. Create a dummy input matching AST's spectrogram shape (1 batch, 1024 time frames, 128 freq bins)
dummy_spectrogram = torch.randn(1, 1024, 128).to(model.device)

# 2. Run the Flop analyzer
print("Calculating AST MACs...")
flops_ast = FlopCountAnalysis(model, dummy_spectrogram)
flops_ast.unsupported_ops_warnings(False)

macs_total = flops_ast.total()
flops_total = macs_total * 2

print("\n" + "="*40)
print(f"AST Computational Complexity")
print("="*40)
print(f"Total MACs (GigaMACs): {macs_total / 1e9:.2f} GMACs")
print(f"Total FLOPs (GigaFLOPs): {flops_total / 1e9:.2f} GFLOPs")
print(f"Total Parameters:      {sum(p.numel() for p in model.parameters()) / 1e6:.2f} M")

In [ ]:
# ── SHOW CLASS-LEVEL REPORT ───────────────────────────────────────────────────

y_pred = np.argmax(results.predictions, axis=1)
y_true = results.label_ids
target_names = [id2label[i] for i in range(10)]
print("\n" + "="*60)
print("CLASS-LEVEL PERFORMANCE REPORT (AST)")
print("="*60)
print(classification_report(y_true, y_pred, target_names=target_names))

In [ ]:
# ── PLOT LOSS CURVES ────────────────────────────────────────────────────────
# extract the internal logs from trainer object
history = trainer_birdclef.state.log_history

train_loss = [x['loss'] for x in history if 'loss' in x]
val_loss = [x['eval_loss'] for x in history if 'eval_loss' in x]

epochs_train = range(1, len(train_loss) + 1)

plt.figure(figsize=(8, 5))
plt.plot(epochs_train, train_loss, label='Training Loss', color='blue')
if len(val_loss) > 0:
    plt.plot(np.linspace(1, len(train_loss), len(val_loss)), val_loss, label='Validation Loss', color='orange', marker='o')
plt.title('AST Fine-Tuning: Loss Curve')
plt.xlabel('Training Steps')
plt.ylabel('Loss')
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
# ── PLOT CONFUSION MATRIX ───────────────────────────────────────────────────
from sklearn.metrics import confusion_matrix

# Get the model's raw predictions on the test dataset
predictions_output = trainer_birdclef.predict(test_dataset)
y_pred = np.argmax(predictions_output.predictions, axis=1)
y_true = predictions_output.label_ids

# Use your unique labels to label the axes
# (Assuming you created 'unique_labels' earlier as a sorted list of your bird names)
unique_labels = sorted(list(set(labels)))

# Calculate the confusion matrix
cm = confusion_matrix(y_true, y_pred)

# Seaborn plot
plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=unique_labels, yticklabels=unique_labels)
plt.title('HuBERT Bird Classification Confusion Matrix')
plt.xlabel('Predicted Bird Species')
plt.ylabel('Actual Bird Species')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()


cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Oranges', xticklabels=unique_labels, yticklabels=unique_labels)
plt.title('AST Bird Classification Confusion Matrix')
plt.xlabel('Predicted Bird Species')
plt.ylabel('Actual Bird Species')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

In [ ]:
# ── AST Demo Setup ────────────────────────────
bird_facts = {
    "barn_swallow": "They are the most widespread swallow in the world, but in America,\n they love building mud nests right on human porches.",
    "common_raven": "Ravens are crazy smart—they actually use logic to solve puzzles and\n have been known to sled down snowy roofs just for fun.",
    "curve_billed_thrasher": "The jazz musicians of the desert; they don't just sing, they\n 'thash' their bills through debris to find insects while singing.",
    "european_starling": "Total acoustics experts! They can mimic human speech, car alarms,\n and even the songs of other birds in your dataset.",
    "house_wren": "Don't let the size fool you; these tiny guys are fiercely territorial and\n will aggressively dive-bomb much larger animals.",
    "northern_cardinal": "Unlike most songbirds, both males and females sing. They use their\n loud, clear whistles to communicate like a high-pitched walkie-talkie.",
    "red_crossbill": "Their beaks are literally crossed! This specialized 'tool' lets them\n pry open pine cones that other birds can't touch.",
    "red_winged_blackbird": "The 'bouncers' of the marsh. Their 'Conk-la-ree!' song is so\n loud and distinct it's usually the first thing our model picks up in noisy data.",
    "song_sparrow": "Every male Song Sparrow has a unique, customized song that he uses to\n defend his territory—like a musical fingerprint.",
    "spotted_towhee": "They are the kings of the 'double-scratch'—a hop-and-kick move they\n use to find food in dead leaves, often making more noise than their song!"
}

classification_tool = pipeline(
    "audio-classification",
    model=model,
    feature_extractor=feature_extractor,
    device=0 if torch.cuda.is_available() else -1
)

def ast_demo(file_path):
    probabilities = classification_tool(file_path)
    result = probabilities[0]
    predicted_label_id = result['label']
    confidence = result['score'] * 100

    # Standardize output string to match dictionary keys (via global dict defined near beginning of file)

    # 1. Get the raw dataset string (e.g., "common_raven")
    raw_bird_name = id2label[int(predicted_label_id.split('_')[-1])] if "LABEL" in predicted_label_id else predicted_label_id
    # 2. Format it for the presentation (e.g., "Common Raven")
    bird_name = raw_bird_name.replace("_", " ").title()

    # format search name differently
    search_name = bird_name.lower().replace(" ", "_")

    print(f"[Bert]: Hey Big A\n")
    time.sleep(1.5)
    print(f"[AST]: Yo bert\n")
    time.sleep(1.5)
    print(f"[Bert]: Yo.\n")
    time.sleep(1.0)
    print(f"[Bert]: I can't tell what this bird is can you help a brotha out\n")
    time.sleep(2.0)
    print(f"[AST]: I'm like {confidence:.1f}% sure that's a {bird_name} big dawg.\n")
    print("-" * 40)

    image_folder = '/content/drive/MyDrive/HuBERT/bird_photos/'
    image_found = False

    if os.path.exists(image_folder):
        # List everything in the folder
        all_files = os.listdir(image_folder)

        # Do a lowercase comparison to ignore weird capitalizations
        for filename in all_files:
            # We check if the raw name (like "common_raven") is anywhere in the filename
            # This catches "Common_Raven.jpg", "common_raven.JPG", "Common raven.jpeg", etc.
            if search_name.replace("_", " ") in filename.lower().replace("_", " "):
                img_p = os.path.join(image_folder, filename)
                display(Image(filename=img_p, width=400))
                image_found = True
                break # Stop searching once we find it

    if not image_found:
        print(f"(bruh the {bird_name} image wasn't found in {image_folder}... sorry twin")

    # For AST, we show the input file + show the input & predicted spectrograms
    time.sleep(2.0)
    print(f"\n[AST]: take your audio file back, ion een want that\n")
    display(Audio(file_path, rate=16000))
    time.sleep(1.5)
    print(f"\n[AST]: Lemme put you on.\n")

    # 1. Generate the Spectrogram for the input audio
    audio, _ = librosa.load(file_path, sr=16000)
    encoded_input = feature_extractor(audio, sampling_rate=16000, return_tensors="np")
    input_spectrogram = encoded_input["input_values"][0] # 2D array input that AST eats up

    # 2. Find a Reference Spectrogram from our test dataset
    ref_spectrogram = None
    for i in range(len(test_dataset)):
        dataset_bird_name = id2label[test_dataset[i]['label']].lower().replace(" ", "_")
        if dataset_bird_name == search_name:
            ref_spectrogram = np.array(test_dataset[i]['input_values'])
            break

    # 3. Plot them side-by-side
    fig, axes = plt.subplots(1, 2, figsize=(14, 4))

    # Plot 1: The Input
    # We use .T (transpose) and origin='lower' to orient time on the X axis and frequency on the Y axis
    axes[0].imshow(input_spectrogram.T, origin='lower', aspect='auto', cmap='magma')
    axes[0].set_title("What AST Saw (Our Input)")
    axes[0].set_ylabel("Frequency Bins")
    axes[0].set_xlabel("Time Frames")

    # Plot 2: The Reference
    if ref_spectrogram is not None:
        axes[1].imshow(ref_spectrogram.T, origin='lower', aspect='auto', cmap='magma')
        axes[1].set_title(f"Reference: {bird_name} (From Training Data)")
        axes[1].set_xlabel("Time Frames")
    else:
        axes[1].text(0.5, 0.5, 'Reference not found', ha='center', va='center')
        axes[1].axis('off')

    plt.tight_layout()
    plt.show()

    time.sleep(2.5)
    fact = bird_facts.get(search_name, "This bird was sponsored as the mascot of Jeffrey Epstein Inc.")
    print(f"\n[AST]: Fun Fact: {fact}\n")
    time.sleep(2.5)
    print(f"[Bert]: U the goat mayn\n")
    time.sleep(2.5)
    print(f"[AST]: Big A out")


In [ ]:
# ── Load Saved AST Model ────────────────────────────
# 1. run !pip install block

# 2. run import block

# 3. run Mount Drive and id2label blocks

# 4. run Load Dataset

# 5. Run Plotting Logic

# 6. Run the ast_demo function above

from transformers import pipeline, ASTForAudioClassification, ASTFeatureExtractor

# 1. Point to the saved model on Drive
saved_model_path = "/content/drive/MyDrive/HuBERT/ast_model_500_10_8_16" # <---- change if need
base_extractor_path = "MIT/ast-finetuned-audioset-10-10-0.4593"

print("Loading saved model from Drive...")
# 2. Load the trained model and the original feature extractor
loaded_model = ASTForAudioClassification.from_pretrained(saved_model_path)
loaded_extractor = ASTFeatureExtractor.from_pretrained(base_extractor_path)

# 3. Rebuild the pipeline for Bert
classification_tool = pipeline(
    "audio-classification",
    model=loaded_model,
    feature_extractor=loaded_extractor,
    device=0 if torch.cuda.is_available() else -1
)

print("AST is online and ready for inference!")

In [ ]:
import torch
import time
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from transformers import Trainer, TrainingArguments
from sklearn.metrics import confusion_matrix, classification_report

# Must be installed: !pip install fvcore
from fvcore.nn import FlopCountAnalysis

print("=" * 60)
print("1. AST ARCHITECTURE & COMPLEXITY (LOADED MODEL)")
print("=" * 60)

# A. Calculate Parameters
num_params = sum(p.numel() for p in loaded_model.parameters())
print(f"Total Parameters: {num_params / 1e6:.2f} Million")

# B. Calculate MACs and FLOPs using a Wrapper
class ModelWrapper(torch.nn.Module):
    def __init__(self, model):
        super().__init__()
        self.model = model
    def forward(self, x):
        return self.model(x).logits

wrapped_model = ModelWrapper(loaded_model).eval()
# AST expects a 2D spectrogram input: [Batch, Time_Frames, Freq_Bins]
dummy_spectrogram = torch.randn(1, 1024, 128).to(loaded_model.device)

flops_ast = FlopCountAnalysis(wrapped_model, dummy_spectrogram)
flops_ast.unsupported_ops_warnings(False)
macs_total = flops_ast.total()

print(f"Total MACs:       {macs_total / 1e9:.2f} GMACs")
print(f"Total FLOPs:      {(macs_total * 2) / 1e9:.2f} GFLOPs")


print("\n" + "=" * 60)
print("2. AST INFERENCE & ACCURACY METRICS")
print("=" * 60)

# C. Setup Trainer for Evaluation
ast_eval_trainer = Trainer(
    model=loaded_model,
    args=TrainingArguments(output_dir="/tmp/ast_eval", per_device_eval_batch_size=16, report_to="none"),
    eval_dataset=test_dataset,
    compute_metrics=compute_metrics,
)

# D. Run Prediction and Time it
start_inference = time.time()
ast_loaded_results = ast_eval_trainer.predict(test_dataset)
end_inference = time.time()

latency_per_sample = ((end_inference - start_inference) / len(test_dataset)) * 1000

print(f"  Accuracy   : {ast_loaded_results.metrics['test_accuracy']:.4f}")
print(f"  F1 Score   : {ast_loaded_results.metrics['test_f1']:.4f}")
print(f"  Precision  : {ast_loaded_results.metrics['test_precision']:.4f}")
print(f"  Recall     : {ast_loaded_results.metrics['test_recall']:.4f}")
print(f"  Latency    : {latency_per_sample:.2f} ms per spectrogram")

# E. Generate Reports and Graphs
y_pred = np.argmax(ast_loaded_results.predictions, axis=1)
y_true = ast_loaded_results.label_ids
unique_labels = sorted(list(set(labels)))
target_names = [id2label[i] for i in range(10)]

print("\n" + "=" * 60)
print("3. CLASS-LEVEL PERFORMANCE REPORT")
print("=" * 60)
print(classification_report(y_true, y_pred, target_names=target_names))

cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Oranges', xticklabels=unique_labels, yticklabels=unique_labels)
plt.title('AST Classification Confusion Matrix (Loaded Model)')
plt.xlabel('Predicted Bird Species')
plt.ylabel('Actual Bird Species')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

In [ ]:
# Run Demo
test_file = '/content/drive/MyDrive/HuBERT/test_audio/neet-code.mp3'
ast_demo(test_file)